In [ ]:
import pathlib
import random
import copy
import gc
import numpy as np 
import torch
import torchvision
import quantus
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt

import os
import mne
from tqdm import tqdm
from scipy.signal import hilbert
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
from tqdm import tqdm

from pycircstat.tests import *
import statsmodels.multivariate.multivariate_ols as mv_ols
import statsmodels.api as sm 
import pandas as pd

In [ ]:
band_combinations = {
    "Theta+Alpha": (4, 13),
    "Alpha+Beta": (8, 30),
    "Beta+Gamma": (13, 45),
    "Theta+Alpha+Beta": (4, 30),
    "Alpha+Beta+Gamma": (8, 45),
    "Theta+Alpha+Beta+Gamma": (4, 45),
}

load_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/evaluation_fROAR"

data_keys = ["trial_indices", "fixed_binary_acc", "rolling_binary_acc"]
all_reps = {}
for key in data_keys:
    all_reps[key] = np.array([])

# Define the directory where the files are located
cwd = os.getcwd()
for subject_index in [2,13,24,27,29,34,42,43,46,60,62,67,69,72,73,80]:
    fig,axs = plt.subplots(nrows=1, ncols=2, figsize=(14,5))
    freq_dict = {}
    for freq_band in band_combinations.keys():
        freq_dict[freq_band] = []
        
        #load file where the the bandpass filter was applied for the neighboring frequency bands (freq_band)
        all_reps= np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{0}_freq_band_interaction_{freq_band}.npz")
        data ={}
        data_mean = {}
        for key in ["trial_indices", "fixed_binary_acc", "rolling_binary_acc"]:
            data[key] = all_reps[key][np.newaxis,]
        
        all_data = []
        for rep in range(1,27):
            new_data = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{rep}_freq_band_interaction_{freq_band}.npz")
            for key in ["trial_indices", "fixed_binary_acc", "rolling_binary_acc"]:
                data[key] = np.vstack((data[key],new_data[key][np.newaxis,]))
                data_mean[key] = np.median(data[key],axis=0)

        
        
        axs[0].plot(data_mean["fixed_binary_acc"], label=f"{freq_band}, mean {np.mean(data_mean['fixed_binary_acc']):.2f}")
        axs[1].plot(data_mean["rolling_binary_acc"], label=f"{freq_band}, mean {np.mean(data_mean['rolling_binary_acc']):.2f}")
        axs[0].set_title(f"Subject {subject_index}, fixed binary acc")
        axs[1].set_title(f"Subject {subject_index}, rolling binary acc")
        
        axs[0].legend()
        axs[1].legend()
    fig.savefig(f"subject_{subject_index}_all_freq_bands.png")
        
        
                #get mean




## test vs performance of top frequency bands  vs beta and gamme individually

# filter out entire data if it contains NaN value

In [ ]:

band_combinations = {
    "beta" : (13, 30),
    "gamma": (30, 45),
    "Beta+Gamma": (13, 45),
    "Alpha+Beta+Gamma": (8, 45),
    "Theta+Alpha+Beta+Gamma": (4, 45),
    "full" :(2,45)
}
data_keys = ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]
all_reps = {}
for key in data_keys:
    all_reps[key] = np.array([])

all_subjects_mean = {si:{acc: {band: 0 for band in band_combinations} for acc in data_keys} for si in [2,13,24,27,29,34,42,43,46,60,62,67,69,72,73,80]}

# Define the directory where the files are located
cwd = os.getcwd()
#for subject_index in [2,13,24,27,29,34,42,43,46]:
for subject_index in [2,13,24,27,29,34,42,43,46,60,62,67,69,72,73,80]:
    fig,axs = plt.subplots(nrows=1, ncols=3, figsize=(14,5))
    freq_dict = {}
    for freq_band in band_combinations.keys():
        freq_dict[freq_band] = []
        
        #load file where the the bandpass filter was applied for the neighboring frequency bands (freq_band)
        if freq_band == "beta" or freq_band == "gamma":
            all_reps = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{0}_freq_band_{freq_band}.npz")
        elif freq_band == "full":
            all_reps = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{0}.npz")
        else:
            all_reps= np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{0}_freq_band_interaction_{freq_band}.npz")
        data ={}
        data_mean = {}
        for key in ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]:
            data[key] = all_reps[key][np.newaxis,]
        
        all_data = []
        if freq_band == "beta" or freq_band == "gamma":
            l = np.arange(1,9)
        else:
            l = np.arange(1,27)
        for rep in l:
            if freq_band == "beta" or freq_band == "gamma":
                new_data = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{rep}_freq_band_{freq_band}.npz")
            elif freq_band == "full":
                new_data = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{rep}.npz")
            else:
                new_data = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{rep}_freq_band_interaction_{freq_band}.npz")
            for key in ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]:
                # filter out new_data if it contains a NAN value
                if np.isnan(new_data[key]).any():
                    continue
                else:
                    data[key] = np.vstack((data[key],new_data[key][np.newaxis,]))
        
        for key in ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]:
            data_mean[key] = np.nanmean(data[key],axis=0)

        for key in data_keys:
            all_subjects_mean[subject_index][key][freq_band] = np.mean(data_mean[key])

        axs[0].plot(data_mean["fixed_binary_acc"], label=f"{freq_band}, mean {np.mean(data_mean['fixed_binary_acc']):.2f}")
        axs[1].plot(data_mean["rolling_binary_acc"], label=f"{freq_band}, mean {np.mean(data_mean['rolling_binary_acc']):.2f}")
        axs[2].plot(data_mean["continuous_finetune_losses"], label=freq_band, alpha=0.5)
        axs[0].set_title(f"Subject {subject_index}, fixed binary acc")
        axs[1].set_title(f"Subject {subject_index}, rolling binary acc")
        axs[2].set_title(f"Subject {subject_index}, continuous finetune losses")
        axs[2].set_yscale("log")
        axs[0].legend()
        axs[1].legend()
    fig.savefig(f"subject_{subject_index}_all_freq_bands_loss.png")
        
        
                #get mean

In [ ]:

band_combinations = {
    "delta" : (2, 4),
    "Delta+Theta": (2, 8),
    "theta" : (4, 8),
    "Theta+Alpha": (4, 13),
    "alpha" : (8, 13),
    "Alpha+Beta": (8, 30),
    "beta" : (13, 30),
    "gamma": (30, 45),
    "Beta+Gamma": (13, 45),
    "Alpha+Beta+Gamma": (8, 45),
    "Theta+Alpha+Beta+Gamma": (4, 45),
    "full" :(2,45)
}
data_keys = ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]
all_reps = {}
for key in data_keys:
    all_reps[key] = np.array([])

all_subjects_mean = {si:{acc: {band: 0 for band in band_combinations} for acc in data_keys} for si in [2,13,24,27,29,34,42,43,46,60,62,67,69,72,73,80]}

# Define the directory where the files are located
cwd = os.getcwd()
#for subject_index in [2,13,24,27,29,34,42,43,46]:
for subject_index in [2,13,24,27,29,34,42,43,46,60,62,67,69,72,73,80]:
    fig,axs = plt.subplots(nrows=1, ncols=3, figsize=(14,5))
    freq_dict = {}
    for freq_band in band_combinations.keys():
        freq_dict[freq_band] = []
        print(freq_band)
        #load file where the the bandpass filter was applied for the neighboring frequency bands (freq_band)
        if freq_band in ["delta", "theta", "alpha", "beta", "gamma"]:
            all_reps = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{0}_freq_band_{freq_band}.npz")
        elif freq_band == "full":
            all_reps = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{0}.npz")
        else:
            all_reps= np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{0}_freq_band_interaction_{freq_band}.npz")
        data ={}
        data_mean = {}
        for key in ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]:
            data[key] = all_reps[key][np.newaxis,]
        
        all_data = []
        if freq_band in ["delta", "theta", "alpha", "beta", "gamma"]:
            l = np.arange(1,9)
        else:
            l = np.arange(1,27)
        for rep in l:
            if freq_band in ["delta", "theta", "alpha", "beta", "gamma"]:
                new_data = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{rep}_freq_band_{freq_band}.npz")
            elif freq_band == "full":
                new_data = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{rep}.npz")
            else:
                new_data = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{rep}_freq_band_interaction_{freq_band}.npz")
            for key in ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]:
                # filter out new_data if it contains a NAN value
                if np.isnan(new_data[key]).any():
                    continue
                else:
                    data[key] = np.vstack((data[key],new_data[key][np.newaxis,]))
        
        for key in ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]:
            data_mean[key] = np.nanmean(data[key],axis=0)

        for key in data_keys:
            all_subjects_mean[subject_index][key][freq_band] = np.mean(data_mean[key])

        axs[0].plot(data_mean["fixed_binary_acc"], label=f"{freq_band}, mean {np.mean(data_mean['fixed_binary_acc']):.2f}")
        axs[1].plot(data_mean["rolling_binary_acc"], label=f"{freq_band}, mean {np.mean(data_mean['rolling_binary_acc']):.2f}")
        axs[2].plot(data_mean["continuous_finetune_losses"], label=freq_band, alpha=0.5)
        axs[0].set_title(f"Subject {subject_index}, fixed binary acc")
        axs[1].set_title(f"Subject {subject_index}, rolling binary acc")
        axs[2].set_title(f"Subject {subject_index}, continuous finetune losses")
        axs[2].set_yscale("log")
        axs[0].legend()
        axs[1].legend()
    fig.savefig(f"subject_{subject_index}_all_freq_bands_loss.png")
        
        
                #get mean

In [ ]:
all_subjects_mean

In [ ]:
import seaborn as sns

for subject_index, metrics in all_subjects_mean.items():
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(18, 6))
    fig.suptitle(f'Subject {subject_index} - Accuracy by Frequency Band', fontsize=20, fontweight='bold')
    
    # Prepare data for seaborn
    bands = list(band_combinations.keys())
    
    for i, key in enumerate(["fixed_binary_acc", "rolling_binary_acc"]):
        # Calculate accuracy gain above random
        values = [metrics[key][band] - 0.5 for band in bands]
        
        # Create seaborn barplot with enhanced styling
        sns.barplot(x=bands, y=values, ax=axs[i], palette="viridis", alpha=0.8)
        
        # Better titles
        title = "Fixed Binary Accuracy" if key == "fixed_binary_acc" else "Rolling Binary Accuracy"
        axs[i].set_title(title, fontsize=18, pad=15)
        axs[i].set_ylabel('Accuracy Gain Above Random (50%)', fontsize=14)
        axs[i].set_xlabel('Frequency Band', fontsize=14)
        axs[i].set_ylim([0, 0.5])
        
        # Rotate x labels
        axs[i].tick_params(axis='x', labelrotation=45, labelsize=12)
        axs[i].tick_params(axis='y', labelsize=12)
        
        # Add grid for better readability
        axs[i].grid(axis='y', linestyle='--', alpha=0.7)
        
        # Format y-ticks as percentages
        axs[i].set_yticklabels([f'{int(tick*100)}%' for tick in axs[i].get_yticks()])
        
        # Add text above each bar - format as percentage
        for j, (p, value) in enumerate(zip(axs[i].patches, values)):
            height = p.get_height()
            axs[i].text(p.get_x() + p.get_width()/2., height + 0.01,
                      f'{int(value*100)}%', 
                      ha='center', fontsize=11, fontweight='bold')
    
    # Add styling touches
    sns.despine(fig=fig)  # Remove top and right spines
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(f'subject_{subject_index}_bar_plot.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'medium',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [ ]:
# Initialize dictionaries to store the mean and std values for each frequency band across all subjects
mean_values = {key: {band: 0 for band in band_combinations.keys()} for key in data_keys}
std_values = {key: {band: 0 for band in band_combinations.keys()} for key in data_keys}

# Calculate the mean and std for each frequency band across all subjects
for key in data_keys:
    for band in band_combinations.keys():
        values = [all_subjects_mean[subject][key][band] for subject in all_subjects_mean.keys()]
        mean_values[key][band] = np.mean(values)
        std_values[key][band] = np.std(values)

# Setup better styling
sns.set_style("whitegrid")

# Create a single figure with two rows
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(16, 12), sharex=True, sharey=True)
fig.suptitle('Mean Performance Across All Subjects', fontsize=24, fontweight='bold', y=0.98)

bands = list(band_combinations.keys())
metrics = ["fixed_binary_acc", "rolling_binary_acc"]
titles = ["Fixed Binary Accuracy", "Rolling Binary Accuracy"]

for i, (metric, title) in enumerate(zip(metrics, titles)):
    # Calculate values for each band (as gain above random chance)
    mean_vals = [mean_values[metric][band] - 0.5 for band in bands]
    std_vals = [std_values[metric][band] for band in bands]
    
    # Create barplot
    bars = axs[i].bar(range(len(bands)), mean_vals, color=sns.color_palette("viridis", n_colors=len(bands)), alpha=0.8)
    
    # Add error bars
    axs[i].errorbar(range(len(bands)), mean_vals, yerr=std_vals, fmt='none', color='black', capsize=5)
    
    # Set x-ticks and labels
    axs[i].set_xticks(range(len(bands)))
    axs[i].set_xticklabels(bands)
    
    # Better styling for axes and labels
    axs[i].set_ylabel('Accuracy Gain Above Random (50%)', fontsize=16)
    axs[i].set_title(title, fontsize=20, fontweight='bold', pad=15)
    axs[i].set_ylim([0, 0.5])
    axs[i].tick_params(axis='x', labelsize=17)
    axs[i].tick_params(axis='y', labelsize=17)
    axs[i].grid(axis='y', linestyle='--', alpha=0.7)
    
    # Format y-ticks as percentages
    axs[i].set_yticklabels([f'{int(tick*100)}%' for tick in axs[i].get_yticks()])
    
    # Add text above each bar showing both mean and std
    for j, bar in enumerate(bars):
        height = bar.get_height()
        axs[i].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{int(mean_vals[j]*100)}% ±{int(std_vals[j]*100)}%', 
                   ha='center', fontsize=12, fontweight='bold')

# Set x-axis label only for the bottom subplot
axs[1].set_xlabel('Frequency Band', fontsize=18, fontweight='bold')

# Rotate x-tick labels for better readability
plt.setp(axs[1].get_xticklabels(), rotation=45, ha='right')

# Remove top and right spines for cleaner look
for ax in axs:
    sns.despine(ax=ax)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('mean_over_subjects_combined_accuracy_bar_plot_with_std.png', dpi=300, bbox_inches='tight')
plt.show()


## with NaN runs included

In [ ]:

band_combinations = {
    "beta" : (13, 30),
    "gamma": (30, 45),
    "Beta+Gamma": (13, 45),
    "Alpha+Beta+Gamma": (8, 45),
    "Theta+Alpha+Beta+Gamma": (4, 45),
    "full" :(2,45)
}
data_keys = ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]
all_reps = {}
for key in data_keys:
    all_reps[key] = np.array([])

# Define the directory where the files are located
cwd = os.getcwd()
#for subject_index in [2,13,24,27,29,34,42,43,46]:
for subject_index in [2,13,24,27,29,34,42,43,46,60,62,67,69,72,73,80]:
    fig,axs = plt.subplots(nrows=1, ncols=3, figsize=(14,5))
    freq_dict = {}
    for freq_band in band_combinations.keys():
        freq_dict[freq_band] = []
        
        #load file where the the bandpass filter was applied for the neighboring frequency bands (freq_band)
        if freq_band == "beta" or freq_band == "gamma":
            all_reps = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{0}_freq_band_{freq_band}.npz")
        elif freq_band == "full":
            all_reps = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{0}.npz")
        else:
            all_reps= np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{0}_freq_band_interaction_{freq_band}.npz")
        data ={}
        data_mean = {}
        for key in ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]:
            data[key] = all_reps[key][np.newaxis,]
        
        all_data = []
        if freq_band == "beta" or freq_band == "gamma":
            l = np.arange(1,9)
        else:
            l = np.arange(1,27)
        for rep in l:
            if freq_band == "beta" or freq_band == "gamma":
                new_data = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{rep}_freq_band_{freq_band}.npz")
            elif freq_band == "full":
                new_data = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{rep}.npz")
            else:
                new_data = np.load(cwd+f"/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}/all_metrics_subject_{subject_index}_rep_{rep}_freq_band_interaction_{freq_band}.npz")
            for key in ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]:
                # filter out new_data if it contains a NAN value
                #if np.isnan(new_data[key]).any():
                #    continue
                #else:
                data[key] = np.vstack((data[key],new_data[key][np.newaxis,]))
        
        for key in ["continuous_finetune_losses", "fixed_binary_acc", "rolling_binary_acc"]:
            data_mean[key] = np.nanmean(data[key],axis=0)

        
        
        axs[0].plot(data_mean["fixed_binary_acc"], label=f"{freq_band}, mean {np.mean(data_mean['fixed_binary_acc']):.2f}")
        axs[1].plot(data_mean["rolling_binary_acc"], label=f"{freq_band}, mean {np.mean(data_mean['rolling_binary_acc']):.2f}")
        axs[0].legend()
        axs[1].legend()
        axs[2].plot(data_mean["continuous_finetune_losses"], label=freq_band, alpha=0.5)
        axs[0].set_title(f"Subject {subject_index}, fixed binary acc")
        axs[1].set_title(f"Subject {subject_index}, rolling binary acc")
        axs[2].set_title(f"Subject {subject_index}, continuous finetune losses")
        axs[2].set_yscale("log")
        plt.legend()
    fig.savefig(f"subject_{subject_index}_all_freq_bands_loss.png")